# Step 3b — Hand-connect the Kabankalan pair

Step 3 left two v1-imported Negros Occidental buses isolated because their nearest existing transmission bus is 33 km away (over the 15 km auto-spur cap):
- `v1_06kabankalan` (Kabankalan substation)
- `v1_06kbanbess`   (Kabankalan BESS, co-located ~400 m from the substation)

Instead of raising the global cap, hand-encode two known connections from the real NGCP Negros topology:

1. **Kabankalan ↔ Mabinay** — 138 kV overhead, 33 km. Mabinay (`v1_06mabinay`) sits in Negros Oriental and is the real neighbour on the Negros transmission backbone. It is already connected to the OSM mesh via a 0.13 km spur to `tower_0048`, so this line bridges Kabankalan into the existing graph.
2. **Kabankalan BESS ↔ Kabankalan** — 138 kV overhead, ~0.4 km. Co-located BESS pigtail.

Both lines tagged `data_source='synthetic_v1_handcoded'` to distinguish them from the MST / spur logic in Step 3 — they reflect named real-grid knowledge, not algorithmic synthesis.

In [1]:
from pathlib import Path
import math
import pandas as pd
import geopandas as gpd
import numpy as np

PROC_DIR = Path('../backend/data/processed')
WGS = 'EPSG:4326'
UTM = 'EPSG:32651'

buses = pd.read_csv(PROC_DIR / 'buses.csv')
lines = pd.read_csv(PROC_DIR / 'lines.csv')
print(f'Inputs: {len(buses)} buses, {len(lines)} lines')

Inputs: 2448 buses, 2450 lines


In [2]:
# Confirm the three target buses exist and grab their coords
TARGETS = ['v1_06kabankalan', 'v1_06kbanbess', 'v1_06mabinay']
tgt = buses[buses['bus_id'].isin(TARGETS)][['bus_id', 'name', 'lat', 'lon', 'voltage_kv', 'province', 'island']]
assert len(tgt) == 3, f'Missing buses; found {len(tgt)}/3'
print(tgt.to_string(index=False))

# Verify Kabankalan + BESS are currently isolated (no incident lines)
incident = lines[(lines['from_bus'].isin(TARGETS)) | (lines['to_bus'].isin(TARGETS))]
print(f'\nIncident lines on the three buses before fix: {len(incident)}')
if len(incident):
    print(incident[['line_id', 'from_bus', 'to_bus', 'voltage_kv', 'length_km']].to_string(index=False))

         bus_id                                          name       lat        lon  voltage_kv          province island
v1_06kabankalan      Kabankalan substation, Negros Occidental 10.019242 122.848185       138.0 Negros Occidental Negros
  v1_06kbanbess Kabankalan BESS substation, Negros Occidental 10.019979 122.851692       138.0 Negros Occidental Negros
   v1_06mabinay           Mabinay substation, Negros Oriental  9.729627 122.925526       138.0   Negros Oriental Negros

Incident lines on the three buses before fix: 1
            line_id     from_bus     to_bus  voltage_kv  length_km
line_synth_spur_012 v1_06mabinay tower_0048       138.0   0.133058


In [3]:
# Compute metric distances in UTM
g = gpd.GeoDataFrame(
    tgt, geometry=gpd.points_from_xy(tgt['lon'], tgt['lat']), crs=WGS
).to_crs(UTM).set_index('bus_id')

d_kab_mab  = g.loc['v1_06kabankalan'].geometry.distance(g.loc['v1_06mabinay'].geometry)
d_kab_bess = g.loc['v1_06kabankalan'].geometry.distance(g.loc['v1_06kbanbess'].geometry)
print(f'Kabankalan ↔ Mabinay:        {d_kab_mab/1000:.2f} km')
print(f'Kabankalan ↔ Kabankalan BESS: {d_kab_bess/1000:.3f} km')

Kabankalan ↔ Mabinay:        33.12 km
Kabankalan ↔ Kabankalan BESS: 0.393 km


In [4]:
# Standard 138 kV overhead impedance (matches Phase 1B + Step 3)
R_138, X_138, IMAX_138 = 0.08, 0.40, 0.70

new_rows = [
    {
        'line_id': 'line_handcoded_kabankalan_mabinay',
        'from_bus': 'v1_06kabankalan',
        'to_bus':   'v1_06mabinay',
        'voltage_kv': 138.0,
        'length_km':  d_kab_mab / 1000,
        'r_ohm_per_km': R_138, 'x_ohm_per_km': X_138, 'max_i_ka': IMAX_138,
        'is_submarine': False, 'cable_type': 'overhead',
        'is_synthetic': True, 'data_source': 'synthetic_v1_handcoded',
    },
    {
        'line_id': 'line_handcoded_kabankalan_bess_pigtail',
        'from_bus': 'v1_06kbanbess',
        'to_bus':   'v1_06kabankalan',
        'voltage_kv': 138.0,
        'length_km':  d_kab_bess / 1000,
        'r_ohm_per_km': R_138, 'x_ohm_per_km': X_138, 'max_i_ka': IMAX_138,
        'is_submarine': False, 'cable_type': 'overhead',
        'is_synthetic': True, 'data_source': 'synthetic_v1_handcoded',
    },
]
new = pd.DataFrame(new_rows)
print(new[['line_id', 'from_bus', 'to_bus', 'voltage_kv', 'length_km', 'data_source']].to_string(index=False))

                               line_id        from_bus          to_bus  voltage_kv  length_km            data_source
     line_handcoded_kabankalan_mabinay v1_06kabankalan    v1_06mabinay       138.0  33.124367 synthetic_v1_handcoded
line_handcoded_kabankalan_bess_pigtail   v1_06kbanbess v1_06kabankalan       138.0   0.392918 synthetic_v1_handcoded


In [5]:
# Align to lines.csv schema, validate, append, save
schema_cols = lines.columns.tolist()
new_aligned = new.reindex(columns=schema_cols, fill_value=np.nan)

valid_buses = set(buses['bus_id'])
assert new_aligned['from_bus'].isin(valid_buses).all()
assert new_aligned['to_bus'].isin(valid_buses).all()
assert (new_aligned['from_bus'] != new_aligned['to_bus']).all()

lines_out = pd.concat([lines, new_aligned], ignore_index=True)
assert lines_out['line_id'].is_unique
lines_out.to_csv(PROC_DIR / 'lines.csv', index=False)
print(f'Wrote {PROC_DIR / "lines.csv"} ({len(lines_out)} rows; +{len(new)} hand-coded)')

Wrote ../backend/data/processed/lines.csv (2452 rows; +2 hand-coded)


## Connectivity verification

In [6]:
import networkx as nx
tx_lines = lines_out[~lines_out['voltage_kv'].between(10, 50)]
tx_buses_in_lines = set(tx_lines['from_bus']) | set(tx_lines['to_bus'])
G = nx.Graph()
G.add_nodes_from(tx_buses_in_lines)
G.add_edges_from(zip(tx_lines['from_bus'], tx_lines['to_bus']))
print(f'Transmission components after hand-connect: {nx.number_connected_components(G)}')
print(f'  Was 18 after Step 3.')

# Which component now contains Kabankalan?
for comp in nx.connected_components(G):
    if 'v1_06kabankalan' in comp:
        print(f'\nKabankalan now sits in a component of size {len(comp)}.')
        # Spot-check 5 members
        sample = list(comp)[:5]
        print(f'  Sample members: {sample}')
        break

# Per-island Negros
negros_buses = set(buses[buses['island'] == 'Negros']['bus_id']) & tx_buses_in_lines
negros_sub = G.subgraph(negros_buses)
print(f'\nNegros: buses={len(negros_buses)}  components={nx.number_connected_components(negros_sub)}  (was 5 after Step 3)')

Transmission components after hand-connect: 18
  Was 18 after Step 3.

Kabankalan now sits in a component of size 92.
  Sample members: ['sub_daanbantayan_cable_terminal_27', 'v1_04kananga', 'sub_suba_cable_terminal_29', 'sub_dumanjug_converter_station_100', 'tower_0025']

Negros: buses=34  components=5  (was 5 after Step 3)
